In [1]:
import sys, os, subprocess
from pathlib import Path

# Colab: clone repo and install deps. Local: resolve root from CWD.
try:
    import google.colab  # noqa
    REPO = '/content/Katabatic'
    if not os.path.exists(REPO):
        subprocess.run(
            ['git', 'clone', '--branch', 'luke', 'https://github.com/lukebrumby/katabatic-personal.git', REPO],
            check=True
        )
    os.chdir(REPO)
    sys.path.insert(0, REPO)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'], check=True)
    ROOT = Path(REPO)
except ImportError:
    ROOT = Path.cwd().resolve()
    for _ in range(5):
        if (ROOT / 'pyproject.toml').exists() or (ROOT / 'raw_data').exists():
            break
        ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

print('ROOT:', ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models_luke.tablegan.models import TableGANModel

ROOT: /content/Katabatic


In [ ]:
TABLEGAN = lambda: TableGANModel(
    z_dim=100,
    num_epochs=200,
    batch_size=500,
    learning_rate=0.0002,
    beta1=0.5,
    alpha=0.5,
    beta=0.5,
    random_state=42,
)

In [3]:
# Preprocess all datasets
DATASETS = ["car", "adult", "magic", "shuttle", "nursery"]

for dataset in DATASETS:
    dataset_path = ROOT / "raw_data" / f"{dataset}.csv"
    output_path = ROOT / "discretized_data" / f"{dataset}.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Preprocessing {dataset}...")
    discretize_preprocess(str(dataset_path), str(output_path))

Preprocessing car...
Preprocessing: /content/Katabatic/raw_data/car.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/car.csv
Preprocessing adult...
Preprocessing: /content/Katabatic/raw_data/adult.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/adult.csv
Preprocessing magic...
Preprocessing: /content/Katabatic/raw_data/magic.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/magic.csv
Preprocessing shuttle...
Preprocessing: /content/Katabatic/raw_data/shuttle.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/shuttle.csv
Preprocessing nursery...
Preprocessing: /content/Katabatic/raw_data/nursery.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/nursery.csv


In [4]:
# Run pipeline on all datasets
for dataset in DATASETS:
    print(f"\n{'='*60}")
    print(f"TableGAN -> {dataset}")
    input_csv = str(ROOT / "discretized_data" / f"{dataset}.csv")
    output_dir = str(ROOT / "sample_data" / dataset)
    real_test_dir = output_dir
    synthetic_dir = str(ROOT / "synthetic" / dataset / "tablegan")

    pipeline = TrainTestSplitPipeline(model=TABLEGAN)
    result = pipeline.run(
        input_csv=input_csv,
        output_dir=output_dir,
        synthetic_dir=synthetic_dir,
        real_test_dir=real_test_dir,
    )
    print(result)


TableGAN -> car
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
[TableGAN] Epoch 50/200  d=1.2568  g=1.4086
[TableGAN] Epoch 100/200  d=1.1243  g=1.3233
[TableGAN] Epoch 150/200  d=1.1911  g=1.0574
[TableGAN] Epoch 200/200  d=1.4256  g=0.9295


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:53:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/car/tablegan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.5665
F1 Score: 0.5992

MLP:
Accuracy: 0.5665
F1 Score: 0.5858

RF:
Accuracy: 0.6127
F1 Score: 0.6399

XGBoost:
Accuracy: 0.6156
F1 Score: 0.6368
Train test split pipeline executed successfully.

TableGAN -> adult
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
[TableGAN] Epoch 50/200  d=1.2393  g=0.5490
[TableGAN] Epoch 100/200  d=1.5515  g=0.6305
[TableGAN] Epoch 150/200  d=1.0816  g=0.6783
[TableGAN] Epoch 200/200  d=0.8507  g=0.9121

Results saved to: Results/adult/tablegan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7737
F1 Score: 0.7393
AUC: 0.7591

MLP:
A

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:17:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/shuttle/tablegan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.9411
F1 Score: 0.9458

MLP:
Accuracy: 0.9946
F1 Score: 0.9963

RF:
Accuracy: 0.9953
F1 Score: 0.9951

XGBoost:
Accuracy: 0.9952
F1 Score: 0.9945
Train test split pipeline executed successfully.

TableGAN -> nursery
Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
0    0.333333
1    0.329186
3    0.312018
4    0.025270
2    0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
0    0.333333
1    0.329090
3    0.312114
4    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)
[TableGAN] Epoch 50/200  d=1.1437  g=0.7622
[TableGAN] Epoch 100/200  d=1.0740  g=0.6797
[TableGAN] Epoch 150/200  d=1.0913  g=0.6212
[TableGAN] Epoch 200/200  d=0.9725  g=0.6532


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:19:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/nursery/tablegan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6821
F1 Score: 0.7242

MLP:
Accuracy: 0.7191
F1 Score: 0.7678

RF:
Accuracy: 0.7253
F1 Score: 0.7557

XGBoost:
Accuracy: 0.7407
F1 Score: 0.7669
Train test split pipeline executed successfully.
